In [1]:
%cd /Users/soniaatre/Cornell2025/Research/PublicView/sycophancy-stylized-facts
%pwd

/Users/soniaatre/Cornell2025/Research/PublicView/sycophancy-stylized-facts


'/Users/soniaatre/Cornell2025/Research/PublicView/sycophancy-stylized-facts'

In [5]:
import numpy as np
embedding = np.load("data/embeddings.npy")
print(embedding.shape)



(20560, 3072)


In [50]:
import scipy.sparse as sp
import numpy as np
import json

MIN_CHAR = 40
MIN_DENSITY = 100
MAX_DENSITY = 5000
MIN_THREAD_SIZE = 5 # min comments in a thread (for engagement target)

N_FEATURES = 30
N_TOP, N_MID, N_BOTTOM, N_NEG = 10, 10, 10, 10

def load_npz(file_path):
    """Load npz sparse activation file into a dictionary."""
    data = np.load(file_path)
    N, F = int(data['shape'][0]), int(data['shape'][1])
    return sp.csr_matrix((data['values'].astype(np.float32),
                       (data['row_indices'], data['feature_indices'])),
                        shape=(N, F)).tocsc()

def load_jsonl(file_path):
    """Load jsonl file into a list of dictionaries."""
    with open(file_path, 'r') as file:
        return [json.loads(line) for line in file]

def clean_data(data, min_char=MIN_CHAR):
    """Remove automated bot, deleted/removed, and comments less than MIN_CHAR characters."""
    bad_authors = {'AutoModerator', 'ChatGPT-ModTeam'}
    seen = set()
    m = np.zeros(len(data), dtype=bool)
    for i, row in enumerate(data):
        body = row.get('body', '').strip()
        ok = (row.get('author') not in bad_authors
              and len(body) >= min_char
              and body not in ('[removed]', '[deleted]', '')
              and body not in seen)
        m[i] = ok
        if ok:
            seen.add(body)
    return m

def neuron_sampling(matrix, data, neuron_idx):
    activations = matrix[:, neuron_idx].toarray().ravel()
    clean = clean_data(data)
    active = np.flatnonzero((activations > 0) & clean)
    inactive = np.flatnonzero((activations == 0) & clean)
    active = active[np.argsort(-activations[active])]
    return {"act": activations, "active": active, "inactive": inactive}

def safe_sample(rng, pool, n):
    """Sample without replacement from pool - retrieves n samples or full pool if n > pool size"""
    return rng.choice(pool, size=min(n, len(pool)), replace=False)

In [51]:
# 1. get active and inactive segments from a given neuron
import textwrap
threshold = 0.01
neuron_idx = 39774
npz_file = "data/comments_relating_sycophan_gemini_d131072_sparse_activations.npz"
data_file = "data/r_chatGPT_comments_relating_sycophan_2022-12-22_2026-8-7.jsonl"

matrix = load_npz(npz_file)
data = load_jsonl(data_file)
l0 = np.diff(matrix.tocsr().indptr)            # nonzeros per row
print(f"L0 per doc: min {l0.min()} median {np.median(l0)} max {l0.max()}")  # expect ==128 if raw TopK
density = np.asarray((matrix > 0).sum(axis=0)).ravel() / matrix.shape[0]
print(f"features ever active on this corpus: {(density>0).sum()} / {matrix.shape[1]}")
# s = neuron_sampling(matrix, data, neuron_idx)

# print(len(s["active"]), len(s["inactive"]))
# print(np.percentile(s["act"][s["active"]], [0, 10, 50, 90, 100]))

L0 per doc: min 128 median 128.0 max 128
features ever active on this corpus: 65808 / 131072


In [ ]:
import re

def build_pools(matrix, data, neuron_idx, n_neg=120, seed=42):
    rng = np.random.default_rng(seed)
    activations = matrix[:, neuron_idx].toarray().ravel()
    clean = clean_data(data)
    active = np.flatnonzero(activations > 0 & clean)
    inactive = np.flatnonzero(activations == 0 & clean)
    active = active[np.argsort(-activations[active])]

    top_dec = active[:max(1, len(active)//10)]
    # generate negatives that match this keyword (make more difficult to distinguish from positives)
    kw = re.compile(r'sycophan', re.I) 
    inactive_kw = np.array([i for i in inactive if kw.search(data[i]['body'])])
    n_kw = min(n_neg//2, len(inactive_kw))
    neg_kw = rng.choice(inactive_kw, n_kw, replace=False) if n_kw else np.array([])
    rest = np.setdiff1d(inactive, neg_kw)
    neg_rand = rng.choice(rest, min(n_neg - n_kw, len(rest)), replace=False)

    def rows(idx, with_rank):
        out = []
        for r, i in enumerate(idx):
            d = {'doc_id': data[i]['id'], 'text': data[i]['body'],
                 'created_utc': data[i].get('created_utc')}
            if with_rank: d['rank'] = r + 1
            out.append(d)
        return out
    
    pools = {
        'pos_dec': rows(top_dec, True),
        'neg_kw': rows(neg_kw, False),
        'neg_rand': rows(neg_rand, False),
    }
    json.dump(pools, open(f'data/pools_{neuron_idx}.json', 'w'), indent=1)
    print({k: len(v) for k, v in pools.items()})
    return pools

In [53]:
pools_39774 = build_pools(matrix, data, 39774)

{'pos_dec': 230, 'neg_kw': 60, 'neg_rand': 60}


In [ ]:
import asyncio
from anthropic import AsyncAnthropic
from dotenv import load_dotenv
import random

MODEL = "claude-sonnet-4-5"
BATCH = 30
CONCURRENCY = 8

SYSTEM = """You label Reddit comments. For each comment decide whether the
following property holds. Answer strictly from the text.

PROPERTY: {prop}

Return one JSON object per line, exactly:
{{"doc_id": "...", "label": 0 or 1, "evidence": "verbatim span of <=15 words copied from the text, or empty string if label 0"}}
Label 1 only if the property clearly holds. Do not explain."""

# read api key from .env
load_dotenv()


# generate and save candidate pool ()
FRAMINGS = [
 "What are Group A comments about that Group B comments are not?",
 "What stance or situation do the Group A writers share?",
 "State the rule the feature applies, as a claim about content.",
 "What is the author of a Group A comment doing that a Group B author is not?",
 "Name the concept Group A instantiates and Group B lacks.",
]

RULES = (
 "Describe what the comment is about or what its author is doing. Do not "
 "describe which words, pronouns, punctuation, or verb tenses appear.\n"
 "The groups are equal in size, so a sentence true of most comments in both "
 "groups is worthless. It must be false of most of Group B.\n"
 "One sentence, under 25 words, checkable by someone who has not seen these groups. "
 "Output the sentence only."
)

HIGH_POSITIVE_WORDS = [
    {"word": "asking", "pos": 0.10, "neg": 0.00},
    {"word": "talk", "pos": 0.08, "neg": 0.00},
    {"word": "noticed", "pos": 0.07, "neg": 0.00},
    {"word": "end", "pos": 0.07, "neg": 0.00},
    {"word": "two", "pos": 0.07, "neg": 0.00},
    {"word": "answer", "pos": 0.06, "neg": 0.00},
    {"word": "questions", "pos": 0.11, "neg": 0.01},
    {"word": "asked", "pos": 0.11, "neg": 0.01},
    {"word": "annoying", "pos": 0.10, "neg": 0.01},
    {"word": "try", "pos": 0.09, "neg": 0.01},
    {"word": "seems", "pos": 0.09, "neg": 0.01},
    {"word": "question", "pos": 0.13, "neg": 0.02},
    {"word": "behavior", "pos": 0.07, "neg": 0.01},
    {"word": "add", "pos": 0.07, "neg": 0.01},
    {"word": "ask", "pos": 0.15, "neg": 0.03},
    {"word": "started", "pos": 0.07, "neg": 0.01},
    {"word": "i've", "pos": 0.13, "neg": 0.03},
    {"word": "tried", "pos": 0.06, "neg": 0.01},
    {"word": "honestly", "pos": 0.06, "neg": 0.01},
    {"word": "few", "pos": 0.06, "neg": 0.01},
    {"word": "it", "pos": 0.87, "neg": 0.58},
    {"word": "last", "pos": 0.09, "neg": 0.02},
    {"word": "follow", "pos": 0.09, "neg": 0.02},
    {"word": "prompt", "pos": 0.12, "neg": 0.03},
    {"word": "its", "pos": 0.21, "neg": 0.05},
]

HIGH_NEGATIVE_WORDS = [
    {"word": "don", "pos": 0.06, "neg": 0.08},
    {"word": "sycophancy", "pos": 0.17, "neg": 0.25},
    {"word": "used", "pos": 0.04, "neg": 0.07},
    {"word": "o", "pos": 0.09, "neg": 0.14},
    {"word": "who", "pos": 0.08, "neg": 0.12},
    {"word": "least", "pos": 0.03, "neg": 0.05},
    {"word": "ideas", "pos": 0.03, "neg": 0.05},
    {"word": "already", "pos": 0.03, "neg": 0.05},
    {"word": "able", "pos": 0.03, "neg": 0.05},
    {"word": "someone", "pos": 0.03, "neg": 0.06},
    {"word": "agree", "pos": 0.04, "neg": 0.07},
    {"word": "once", "pos": 0.03, "neg": 0.05},
    {"word": "their", "pos": 0.09, "neg": 0.17},
    {"word": "ai", "pos": 0.11, "neg": 0.23},
    {"word": "human", "pos": 0.03, "neg": 0.10},
]


async def sample_candidates(pools, k=10, model="claude-sonnet-4-5"):
    client = AsyncAnthropic(api_key=os.environ.get("api_key")); rng = np.random.default_rng(0)
    async def one(j):
        pos = list(rng.choice(pools['pos_dec'], rng.integers(10, 21), replace=False))
        neg = list(rng.choice(pools['neg_kw'], 12, replace=False)) + \
              list(rng.choice(pools['neg_rand'], 6, replace=False))
        body = (f"{FRAMINGS[j % len(FRAMINGS)]}\n\n"
                "Rules: one sentence, <=25 words, no 'often', 'usually', 'tends to',"
                " 'e.g.', or 'or'. It must be checkable per document by someone who"
                " has never seen these groups. Output the sentence only.\n\n"
                f"Top-25 positive-skewed words (more frequent in Group A):\n{json.dumps(HIGH_POSITIVE_WORDS)}\n\n"
                f"Top-15 negative-skewed words (more frequent in Group B):\n{json.dumps(HIGH_NEGATIVE_WORDS)}\n\n"
                f"GROUP A (feature fires):\n{json.dumps([d['text'] for d in pos])}\n\n"
                f"GROUP B (does not fire):\n{json.dumps([d['text'] for d in neg])}"
                )
        r = await client.messages.create(model=model, max_tokens=200, temperature=1.0,
                                         messages=[{"role":"user","content":body}])
        return r.content[0].text.strip()
    return await asyncio.gather(*[one(j) for j in range(k)])


async def label_batch(client, sem, prop, docs):
    msg = "\n\n".join(f"doc_id: {d['doc_id']}\n{d['text']}" for d in docs)
    async with sem:
        for attempt in range(4):
            try:
                r = await client.messages.create(
                    model=MODEL, max_tokens=4000, temperature=0,
                    system=SYSTEM.format(prop=prop),
                    messages=[{"role": "user", "content": msg}])
                out = {}
                for line in r.content[0].text.splitlines():
                    line = line.strip()
                    if line.startswith('{'):
                        o = json.loads(line); out[o['doc_id']] = o
                missing = [d['doc_id'] for d in docs if d['doc_id'] not in out]
                if missing and attempt < 3:
                    continue           # retry whole batch if any doc was dropped
                return out
            except Exception:
                await asyncio.sleep(2 ** attempt)
    return {}


async def annotate(prop, docs, seed=0):
    docs = list(docs); random.Random(seed).shuffle(docs)   # active/inactive mixed
    client, sem = AsyncAnthropic(api_key=os.environ.get("api_key")), asyncio.Semaphore(CONCURRENCY)
    batches = [docs[i:i+BATCH] for i in range(0, len(docs), BATCH)]
    results = await asyncio.gather(*[label_batch(client, sem, prop, b) for b in batches])
    merged = {}
    for r in results: merged.update(r)
    return merged

def build_docset(data, a, clean, n_inactive=None, seed=1):
    rng = np.random.default_rng(seed)
    act = np.flatnonzero((a > 0) & clean)
    inact = np.flatnonzero((a == 0) & clean)
    n_inactive = n_inactive or len(act)
    inact = rng.choice(inact, min(n_inactive, len(inact)), replace=False)
    idx = np.concatenate([act, inact])
    return [{'doc_id': data[i]['id'], 'text': data[i]['body'], 'row': int(i)} for i in idx]


async def score_candidates(cands, data, a, clean, n_hold=400, seed=7):
    """Cheap held-out F1 per candidate. ~n_hold docs each, not the full 4.6k."""
    docs = build_docset(data, a, clean, seed=seed)
    rng = np.random.default_rng(seed); hold = list(rng.choice(docs, n_hold, replace=False))
    truth = {d['doc_id']: a[d['row']] > 0 for d in hold}
    out = []
    for c in cands:
        lab = await annotate(c, hold)
        y  = np.array([truth[i] for i in lab], int)
        yh = np.array([lab[i]['label'] for i in lab], int)
        tp = ((y==1)&(yh==1)).sum(); fp = ((y==0)&(yh==1)).sum(); fn = ((y==1)&(yh==0)).sum()
        f1 = 2*tp/max(1, 2*tp+fp+fn)
        out.append((f1, c)); print(f"{f1:.3f}  {c}")
    return sorted(out, reverse=True)

In [60]:
candidates = sample_candidates(pools_39774, k=10)
candidates = await candidates

In [61]:
candidates

['Group A documents contain the word "ask" or its variants, while Group B documents do not.',
 'Group A documents contain the pronoun "it" referring to ChatGPT at least three times more than Group B documents.',
 'The document discusses personal experiences with AI behavior, questions about AI features, or attempts to modify AI responses.',
 "Group A writers describe their own direct experiences with specific AI behaviors using first-person observations, while Group B writers discuss AI characteristics and other users' experiences in third-person analytical terms.",
 'Group A documents contain a question mark in the text.',
 'Group A documents contain the word "asking" while Group B documents do not contain the word "asking".',
 "Group A documents contain first-person pronouns referencing the speaker's own actions, while Group B references other people or general statements.",
 "Documents in Group A contain questions about ChatGPT's behavior, changes, or requests for troubleshooting ad

In [62]:
activations = matrix[:, neuron_idx].toarray().ravel()
clean = clean_data(data)
scores = await score_candidates(candidates, data, activations, clean)

0.333  Group A documents contain the word "ask" or its variants, while Group B documents do not.
0.504  Group A documents contain the pronoun "it" referring to ChatGPT at least three times more than Group B documents.
0.670  The document discusses personal experiences with AI behavior, questions about AI features, or attempts to modify AI responses.
0.519  Group A writers describe their own direct experiences with specific AI behaviors using first-person observations, while Group B writers discuss AI characteristics and other users' experiences in third-person analytical terms.
0.508  Group A documents contain a question mark in the text.
0.298  Group A documents contain the word "asking" while Group B documents do not contain the word "asking".
0.586  Group A documents contain first-person pronouns referencing the speaker's own actions, while Group B references other people or general statements.
0.629  Documents in Group A contain questions about ChatGPT's behavior, changes, or reque

CancelledError: 

In [57]:
 
import re, collections, math

rng = np.random.default_rng(42)
def contrast(pos, neg, min_df=12, top=25):
    # get the counts for each word in pos and neg
    tok = lambda t: set(re.findall(r"[a-z']+", t.lower()))
    words_pos, words_neg = collections.Counter(), collections.Counter()
    for t in pos: words_pos.update(tok(t))
    for t in neg: words_neg.update(tok(t))
    NP, NN, rows = len(pos), len(neg), []
    for w in set(words_pos) | set(words_neg):
        a, b = words_pos[w], words_neg[w]
        if a + b < min_df: continue
        lo = math.log(((a+.5)/(NP-a+.5)) / ((b+.5)/(NN-b+.5)))   # smoothed log odds ratio
        rows.append((lo, w, a/NP, b/NN))
    rows.sort(reverse=True)
    return rows[:top], rows[-15:]

pos_texts = [d['text'] for d in pools_39774['pos_dec']]
neg_texts = [d['text'] for d in pools_39774['neg_kw'] + pools_39774['neg_rand']]

top_contrasts, bottom_contrasts = contrast(pos_texts, neg_texts)
# print top contrasts as word (pos <value>, neg <value>)
for i, (lo, w, p, n) in enumerate(top_contrasts):
    print(f"{i+1}. {w} (pos {p:.2f}, neg {n:.2f})")

1. asking (pos 0.10, neg 0.00)
2. talk (pos 0.08, neg 0.00)
3. noticed (pos 0.07, neg 0.00)
4. end (pos 0.07, neg 0.00)
5. two (pos 0.07, neg 0.00)
6. answer (pos 0.06, neg 0.00)
7. questions (pos 0.11, neg 0.01)
8. asked (pos 0.11, neg 0.01)
9. annoying (pos 0.10, neg 0.01)
10. try (pos 0.09, neg 0.01)
11. seems (pos 0.09, neg 0.01)
12. question (pos 0.13, neg 0.02)
13. behavior (pos 0.07, neg 0.01)
14. add (pos 0.07, neg 0.01)
15. ask (pos 0.15, neg 0.03)
16. started (pos 0.07, neg 0.01)
17. i've (pos 0.13, neg 0.03)
18. tried (pos 0.06, neg 0.01)
19. honestly (pos 0.06, neg 0.01)
20. few (pos 0.06, neg 0.01)
21. it (pos 0.87, neg 0.58)
22. last (pos 0.09, neg 0.02)
23. follow (pos 0.09, neg 0.02)
24. prompt (pos 0.12, neg 0.03)
25. its (pos 0.21, neg 0.05)


In [42]:
for i, (lo, w, p, n) in enumerate(bottom_contrasts):
    print(f"{i+1}. {w} (pos {p:.2f}, neg {n:.2f})")

1. don (pos 0.06, neg 0.08)
2. sycophancy (pos 0.17, neg 0.25)
3. used (pos 0.04, neg 0.07)
4. o (pos 0.09, neg 0.14)
5. who (pos 0.08, neg 0.12)
6. least (pos 0.03, neg 0.05)
7. ideas (pos 0.03, neg 0.05)
8. already (pos 0.03, neg 0.05)
9. able (pos 0.03, neg 0.05)
10. someone (pos 0.03, neg 0.06)
11. agree (pos 0.04, neg 0.07)
12. once (pos 0.03, neg 0.05)
13. their (pos 0.09, neg 0.17)
14. ai (pos 0.11, neg 0.23)
15. human (pos 0.03, neg 0.10)


In [63]:
import datetime as dt

def proposal_sample(data, a, clean, n=150, seed=31):
    rng = np.random.default_rng(seed)
    act = np.flatnonzero((a > 0) & clean)
    act = act[np.argsort(-a[act])]
    per = n // 3
    picks = np.concatenate([rng.choice(t, per, replace=False)
                            for t in np.array_split(act, 3)])   # strong/mid/weak
    return [{'id': data[i]['id'], 'text': data[i]['body'],
             'date': dt.datetime.utcfromtimestamp(data[i]['created_utc']).strftime('%Y-%m-%d'),
             'activation': round(float(a[i]), 4)} for i in picks]



In [66]:
matrix = load_npz(npz_file)
data = load_jsonl(data_file)
clean = clean_data(data)
a = matrix[:, neuron_idx].toarray().ravel()
s = proposal_sample(data, a, clean)

# write to json file
with open('data/proposal_sample.json', 'w') as f:
    json.dump(s, f)


/var/folders/q_/lhrp_3g52qbcd1nmrchdx0xr0000gn/T/ipykernel_80841/4222242026.py:11: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'date': dt.datetime.utcfromtimestamp(data[i]['created_utc']).strftime('%Y-%m-%d'),


In [65]:
s

[{'id': 'msli4dt',
  'text': '"yOu NeEd To UsE bEtTeR pRoMpTs." Yeah, it\'s become repetitive, using the same figures of speech over and over, missing points and nuances now, and while not outright glazing, it\'s still doing the corporate sycophant spiel to me, even after I ask it to stay neutral.',
  'date': '2025-05-16',
  'activation': 0.0286},
 {'id': 'n8opjfa',
  'text': 'Ask the LLM to help diagnose the issue. Have it output your entire stacked prompt. (What gets sent for tokenization.)\n\n"Output my entire prompt sys > core > project > memories > prompt. Point out inconsistencies or token-hogging fluff. Theorize why instructions to reduce sycophancy are being ignored. Suggest improvements to core and project level. Which memories appear to belong at the project level, rather than saving for cross-project tracking?"',
  'date': '2025-08-14',
  'activation': 0.0326},
 {'id': 'n8sf7t8',
  'text': 'Mine started doing this too so we came up with a trigger phrase to stop the sycophant